# NeuroMM-2026 — EEGMamba End-to-End Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/neuromm2026/blob/main/neuromm2026_pipeline.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hashirama21/neuromm2026/blob/main/neuromm2026_pipeline.ipynb)

This notebook walks through the complete NeuroMM-2026 pipeline — from data acquisition to leaderboard submission — using the **EEGMamba Y-Architecture**.

---

## Contents

| # | Section | Description |
|:--|:--------|:------------|
| 1 | [Environment](#1--environment) | Installation and hardware verification |
| 2 | [Dataset](#2--dataset) | HuggingFace download, validation, and exploration |
| 3 | [Configuration](#3--configuration) | YAML config system and per-track overrides |
| 4 | [Architecture](#4--architecture) | Model construction, parameter budget, forward-pass audit |
| 5 | [Smoke Test](#5--smoke-test) | Pipeline validation on a 10 % patient subset |
| 6 | [Pre-training](#6--pre-training) | Multi-task backbone pre-training |
| 7 | [Track 1 — EEG only](#7--track-1--eeg-only) | Binary spike detection · AUPRC |
| 8 | [Track 2 — EEG + Video](#8--track-2--eeg--video) | Uncertainty-aware gating · AUPRC |
| 9 | [Track 3 — Localisation](#9--track-3--localisation) | 5-class epileptogenic zone · Weighted-F1 |
| 10 | [Evaluation & Calibration](#10--evaluation--calibration) | AUPRC / Weighted-F1 · Isotonic calibration |
| 11 | [Inference & Submission](#11--inference--submission) | Candidate set ensemble → Codabench ZIP |
| 12 | [Unit Tests](#12--unit-tests) | Full pytest suite |

---

**Challenge:** [NeuroMM-2026](https://2026.neuromm.org)  
**Dataset:** 25,426 EEG windows (29 ch × 2,000 pts @ 500 Hz) + pre-extracted features from 7 vision backbones  
**Metrics:** AUPRC (Track 1/2) · Weighted-F1 (Track 3)  
**Leaderboard:** [Codabench](https://2026.neuromm.org)

---
## 1 · Environment

Install all dependencies and verify the hardware configuration. The notebook automatically selects the optimal batch size based on available GPU memory.

> **Execution order:** Run cells sequentially from top to bottom. If you skip Section 6 (pre-training), set `BACKBONE_CKPT` manually before running Track sections.


In [ ]:
# Core dependencies
!pip install -q \
    "torch>=2.1.0" "numpy>=1.24.0" "pandas>=2.0.0" \
    "scikit-learn>=1.3.0" "scipy>=1.11.0" "pyyaml>=6.0" \
    "tqdm>=4.65.0" "pytest>=7.4.0" "huggingface_hub>=0.21.0" \
    matplotlib

# Optional — real Mamba2 CUDA kernels (4–8× faster than the native fallback)
# Requires CUDA 11.8+ and matching torch build
# !pip install -q mamba-ssm causal-conv1d

# Optional — LoRA fine-tuning (only when --foundation_ckpt is used)
# !pip install -q peft

# Optional — experiment tracking
# !pip install -q wandb

In [ ]:
import os, sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    _gpu = torch.cuda.get_device_properties(0)
    VRAM_GB = _gpu.total_memory / 1024 ** 3
    BATCH_SIZE = 256 if VRAM_GB > 20 else 128  # L4 ≥ 24 GB / T4 16 GB
    print(f"GPU   : {_gpu.name}")
    print(f"VRAM  : {VRAM_GB:.1f} GB")
else:
    BATCH_SIZE = 16
    print("Device : CPU  (smoke-test recommended; full training will be slow)")

print(f"PyTorch    : {torch.__version__}")
print(f"Device     : {DEVICE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Project    : {PROJECT_ROOT}")

# Fallback — overridden by inspect-ckpt (Section 6) if pre-training is run
BACKBONE_CKPT = "checkpoints/pretrain/best_backbone.pt"


---
## 2 · Dataset

The dataset is hosted on HuggingFace and requires gated access.  
Request access at [NeuroMM/NeuroMM-2026](https://huggingface.co/datasets/NeuroMM/NeuroMM-2026).

```
NeuroMM-2026/
├── annotations/neuromm2026_train_val.csv   # 25,426 rows
└── archives/
    ├── eeg.tar                             # (29, 2000) float32
    ├── video_videomae-large.tar            # (1, 1024)
    ├── video_dinov2-large.tar              # (8, 1024)
    └── ...                                 # 5 further backbones
```

`prepare_data.py` orchestrates five sequential steps: **download → extract → validate → index → smoke subset**.

In [ ]:
# Provide your HuggingFace token via environment variable (recommended)
# or set it directly below (do not commit)
#
#   export HF_TOKEN=hf_XXXX

HF_TOKEN = os.getenv("HF_TOKEN", "")

if HF_TOKEN:
    print(f"HF token : {HF_TOKEN[:8]}...")
else:
    raise EnvironmentError(
        "HF_TOKEN is not set.\n"
        "Set it with: export HF_TOKEN=hf_XXXX, then re-run this cell."
    )

In [ ]:
# Download EEG + VideoMAE-Large + DINOv2-Large (sufficient for all three tracks)
!python scripts/prepare_data.py \
    --hf_token {HF_TOKEN} \
    --backbones videomae-large dinov2-large

# All 7 backbones + candidate set:
# !python scripts/prepare_data.py --hf_token {HF_TOKEN} --backbones all --include_candidate

# Re-run validation and indexing only (archives already on disk):
# !python scripts/prepare_data.py --skip_download --skip_extract

In [ ]:
df = pd.read_csv("annotations/neuromm2026_train_val.csv")

print(f"Total samples : {len(df):,}")
print(f"Subjects      : {df['subject_id'].nunique()}")
print()

summary = pd.DataFrame({
    "Split"     : df["split"].value_counts(),
    "Positive"  : df.groupby("split")["label"].sum().astype(int),
    "Negative"  : df.groupby("split")["label"].apply(lambda x: (x == 0).sum()),
    "Pos rate"  : df.groupby("split")["label"].mean().round(3),
})
print(summary.to_string())

print("\nlabel_type distribution (0 = negative, 1–5 = seizure subtype):")
print(df["label_type"].value_counts().sort_index().to_string())

df_val = df[df['split'] == 'val'].reset_index(drop=True)
print(f'\nValidation set : {len(df_val):,} samples')


In [ ]:
sample_id = df.iloc[0]["sample_id"]
eeg = np.load(f"processed/features/eeg/{sample_id}.npy")

print(f"EEG  shape : {eeg.shape}  —  29 channels × 2,000 pts @ 500 Hz = 4 s")
print(f"     dtype : {eeg.dtype}")
print(f"     range : [{eeg.min():.4f}, {eeg.max():.4f}]")

try:
    vid = np.load(f"processed/features/video/dinov2-large/{sample_id}.npy")
    print(f"\nDINOv2-L shape : {vid.shape}  —  8 frames × 1,024 dim")
    print(f"         dtype : {vid.dtype}")
except FileNotFoundError:
    print("\nVideo features not yet available (run prepare_data.py first).")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

pos_id = df[df["label"] == 1].iloc[0]["sample_id"]
neg_id = df[df["label"] == 0].iloc[0]["sample_id"]

fig = plt.figure(figsize=(16, 7))
fig.suptitle("EEG Windows — Positive (seizure) vs Negative (background)",
             fontsize=13, fontweight="bold", y=1.01)
gs = gridspec.GridSpec(1, 2, wspace=0.05)
t  = np.arange(2000) / 500.0

for col, (sid, title, color) in enumerate([
    (pos_id, "Positive — spike/seizure", "#c0392b"),
    (neg_id, "Negative — background",   "#2980b9"),
]):
    ax  = fig.add_subplot(gs[col])
    eeg = np.load(f"processed/features/eeg/{sid}.npy")
    off = np.arange(29) * 6
    for c in range(29):
        ax.plot(t, eeg[c] + off[c], color=color, lw=0.5, alpha=0.75)
    ax.set_title(title, fontsize=11, color=color)
    ax.set_xlabel("Time (s)")
    ax.set_yticks(off[::4])
    ax.set_yticklabels([f"Ch {c+1}" for c in range(0, 29, 4)], fontsize=8)
    ax.grid(axis="x", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    if col:
        ax.set_yticklabels([])

Path("logs").mkdir(exist_ok=True)
plt.tight_layout()
plt.savefig("logs/eeg_samples.png", dpi=120, bbox_inches="tight")
plt.show()


---
## 3 · Configuration

Configurations follow a **base-inherit pattern**: `base.yaml` defines shared hyperparameters; track-specific files extend and override only what differs.

```
configs/
├── base.yaml      backbone · pre-training · augmentation · logging
├── track1.yaml    head · loss · training schedule  (EEG only)
├── track2.yaml    + video backbones · gating · cross-attention
└── track3.yaml    + n_classes=5 · WeightedCE · channel attention
```

In [ ]:
from src.utils.config import load_config

cfg       = load_config("configs/base.yaml")
cfg_t1    = load_config("configs/track1.yaml")
cfg_t2    = load_config("configs/track2.yaml")
cfg_t3    = load_config("configs/track3.yaml")

In [ ]:
def _fmt(d: dict, indent: int = 2) -> str:
    pad = " " * indent
    return "\n".join(f"{pad}{k}: {v}" for k, v in d.items())

print("── Backbone (shared) ────────────────────────")
print(_fmt(cfg["backbone"]))

print("\n── Pre-training ─────────────────────────────")
print(_fmt(cfg["pretrain"]))

print("\n── Augmentation ─────────────────────────────")
for k, v in cfg["augmentation"].items():
    print(f"  {k}: {v}")

print("\n── Track overrides ──────────────────────────")
rows = [
    ["Track",         "T1",  "T2",  "T3"],
    ["use_video",     cfg_t1.get("use_video"), cfg_t2.get("use_video"), cfg_t3.get("use_video")],
    ["metric",        cfg_t1.get("metric"), cfg_t2.get("metric"), cfg_t3.get("metric")],
    ["epochs",        cfg_t1["training"]["epochs"], cfg_t2["training"]["epochs"], cfg_t3["training"]["epochs"]],
    ["head lr",       cfg_t1["training"]["lr"],     cfg_t2["training"]["lr"],     cfg_t3["training"]["lr"]],
    ["backbone lr",   cfg_t1["training"]["backbone_lr"], cfg_t2["training"]["backbone_lr"], cfg_t3["training"]["backbone_lr"]],
]
col_w = [20, 12, 12, 12]
for r in rows:
    print("".join(str(v).ljust(w) for v, w in zip(r, col_w)))

---
## 4 · Architecture

The **Y-Architecture** shares a single EEGMamba backbone across all three tracks.

```
EEG  (B, 29, 2000)
        │
   LocalCNN        per-channel 1D-CNN   → spike morphology (20–70 ms)
        │          (B, 29, 64)
   DynamicGAT      multi-head self-attn over channel tokens
        │          learned adjacency — not hard-wired 10-20 topology
        │          (B, 29, 128)
   EEGMamba        Mamba2 SSM, O(n), 4 layers, d_model = 256
        │          (B, 256)
        ├── Track1Head   MLP binary          → AUPRC
        ├── Track2Head   MC-Dropout gate
        │                + Cross-Attention   → AUPRC
        └── Track3Head   ChannelTemporalAttn
                         + Video late fusion → Weighted-F1
```

In [ ]:
from src.models import build_model_from_config

model = build_model_from_config(cfg).to(DEVICE)
counts = model.count_parameters()

print(f"{'Module':<24} {'Parameters':>12}")
print("-" * 38)
for k, v in counts.items():
    if k != "total":
        print(f"  {k:<22} {v:>12,}")
print("-" * 38)
print(f"  {'TOTAL':<22} {counts['total']:>12,}")

In [ ]:
B, C, T = 4, 29, 2000

dummy_eeg = torch.randn(B, C, T, device=DEVICE)
dummy_vid = {
    "videomae-large": torch.randn(B, 1, 1024, device=DEVICE),
    "dinov2-large":   torch.randn(B, 8, 1024, device=DEVICE),
}

model.eval()
with torch.no_grad():
    out1 = model(dummy_eeg, mode="track1")
    out2 = model(dummy_eeg, video_features=dummy_vid, mode="track2")
    out3 = model(dummy_eeg, video_features=dummy_vid, mode="track3")

    masked = dummy_eeg.clone()
    masked[:, :, :400] = 0.0
    pt = model(dummy_eeg, mode="pretrain", masked_eeg=masked)

checks = [
    ("track1  logit",         out1["logit"].shape,         (B,)),
    ("track2  logit",         out2["logit"].shape,         (B,)),
    ("track2  gate",          out2["gate"].shape,          (B,)),
    ("track3  logit",         out3["logit"].shape,         (B, 5)),
    ("pretrain binary",       pt["binary_logit"].shape,    (B,)),
    ("pretrain label_type",   pt["labeltype_logit"].shape, (B, 6)),
    ("pretrain recon",        pt["recon"].shape,           (B, C, T)),
]

print(f"{'Output':<28} {'Shape':>20}  {'Status':>8}")
print("-" * 62)
for name, shape, expected in checks:
    status = "PASS" if tuple(shape) == expected else f"FAIL (expected {expected})"
    print(f"  {name:<26} {str(tuple(shape)):>20}  {status:>8}")

In [ ]:
from src.training.losses import FocalPolyLoss

model.train()
loss_fn = FocalPolyLoss()

eeg  = torch.randn(B, C, T, device=DEVICE)
tgt  = torch.randint(0, 2, (B,), device=DEVICE).float()
out  = model(eeg, mode="track1")
loss = loss_fn(out["logit"], tgt)
loss.backward()

n_grads = sum(1 for p in model.parameters() if p.grad is not None)
n_nan   = sum(1 for p in model.parameters() if p.grad is not None and p.grad.isnan().any())

print(f"Parameters with gradient : {n_grads}")
print(f"NaN gradients            : {n_nan}")
print(f"Loss value               : {loss.item():.4f}")
print(f"Gradient check           : {'PASS' if n_nan == 0 else 'FAIL'}")

---
## 5 · Smoke Test

Run this before any full training. The smoke test executes the complete pipeline on a **10 % patient-stratified subset**, covering data loading, augmentation, forward/backward pass, loss computation, and checkpoint saving.

| Hardware | Batch | Est. duration (2 epochs) |
|:---------|------:|:------------------------|
| CPU      |    16 | ~15 min                 |
| T4       |   128 | ~2 min                  |
| L4       |   256 | ~1 min                  |

In [ ]:
import subprocess

smoke_csv = Path("annotations/neuromm2026_smoke_10pct.csv")

if not smoke_csv.exists():
    subprocess.run(
        [sys.executable, "scripts/prepare_data.py",
         "--skip_download", "--skip_extract", "--smoke_pct", "0.10"],
        check=True,
    )

df_smoke = pd.read_csv(smoke_csv)
print(f"Smoke subset : {len(df_smoke):,} samples")
print(df_smoke.groupby(["split", "label"]).size().unstack(fill_value=0).to_string())


In [ ]:
!python scripts/pretrain.py \
    --config configs/base.yaml \
    --annotations annotations/neuromm2026_smoke_10pct.csv \
    --max_epochs 2 \
    --batch_size {BATCH_SIZE} \
    --num_workers 4

ckpt = Path("checkpoints/pretrain/best_backbone.pt")
status = f"{ckpt.stat().st_size / 1024**2:.1f} MB" if ckpt.exists() else "NOT FOUND"
print(f"\nBackbone checkpoint : {status}")

---
## 6 · Pre-training

The backbone is trained with three simultaneous objectives:

| Objective | Loss | Weight | Purpose |
|:----------|:-----|-------:|:--------|
| Binary detection | FocalPolyLoss | 1.0 | Discriminate spike / non-spike |
| Masked reconstruction | MSE | 0.3 | Robust temporal representation |
| Label-type classification | CrossEntropy | 0.5 | Enrich embedding for Track 3 |

AMP (`float16`) is enabled automatically on GPU, halving memory usage and doubling throughput.

In [ ]:
!python scripts/pretrain.py \
    --config configs/base.yaml \
    --batch_size {BATCH_SIZE} \
    --num_workers 4

# Warm-start from a pre-trained EEGMamba foundation checkpoint (LoRA adapters
# are added automatically when --foundation_ckpt is provided):
# !python scripts/pretrain.py \
#     --config configs/base.yaml \
#     --foundation_ckpt pretrained/eegmamba_base.pt \
#     --batch_size {BATCH_SIZE}

In [ ]:
ckpt_path = Path("checkpoints/pretrain/best_backbone.pt")
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"Checkpoint  : {ckpt_path}")
print(f"Epoch       : {ckpt.get('epoch', 'N/A')}")
print(f"Best loss   : {ckpt.get('loss', float('nan')):.4f}")
print(f"Keys        : {list(ckpt.keys())}")
print(f"Tensors     : {len(ckpt.get('backbone_state', {}))}")

BACKBONE_CKPT = str(ckpt_path)

---
## 7 · Track 1 — EEG only

**Task:** Binary spike/seizure detection from EEG alone.  
**Metric:** AUPRC — area under the precision-recall curve.  

**Training strategy:**
- 5-fold patient-disjoint CV (`GroupKFold` on `subject_id`)
- `RobustScaler` fitted on fold-train only (median + IQR, robust to artefact spikes)
- `WeightedRandomSampler` — negative:positive ratio capped at 4:1
- `FocalPolyLoss` (γ = 2, α = 0.75, ε = 1.0)
- `SequentialLR`: linear warm-up → cosine annealing
- Isotonic regression calibration on out-of-fold scores

In [ ]:
!python scripts/train_track.py \
    --track 1 \
    --config configs/track1.yaml \
    --backbone_ckpt {BACKBONE_CKPT}

In [ ]:
import pickle
from src.data.dataset import build_dataloader
from src.data.preprocessing import load_scalers
from src.evaluation.metrics import compute_auprc, compute_auroc

fold_scores, val_labels = [], None

for fold in range(5):
    ckpt_path   = Path(f"checkpoints/track1/fold{fold}.pt")
    scaler_path = Path(f"checkpoints/track1/scalers_fold{fold}.pkl")
    if not ckpt_path.exists():
        continue

    scalers = load_scalers(scaler_path)
    loader  = build_dataloader(
        df_val, Path("processed/features/eeg"), scalers,
        track=1, batch_size=256, augment=False, shuffle=False, num_workers=4,
    )
    m = build_model_from_config(cfg_t1).to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False)["model_state"])
    m.eval()

    logits, labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits.append(torch.sigmoid(m(batch["eeg"].to(DEVICE), mode="track1")["logit"]).cpu())
            labels.append(batch["label"])

    s = torch.cat(logits).numpy()
    t = torch.cat(labels).numpy()
    if val_labels is None:
        val_labels = t
    fold_scores.append(s)
    print(f"  Fold {fold}  AUPRC = {compute_auprc(t, s):.4f}")

if fold_scores:
    ens = np.mean(fold_scores, axis=0)
    calib_path = Path("checkpoints/track1/calibrator.pkl")
    if calib_path.exists():
        with open(calib_path, "rb") as f:
            calib = pickle.load(f)
        ens_cal = calib.transform(ens)
        print(f"\nEnsemble AUPRC (raw)       : {compute_auprc(val_labels, ens):.4f}")
        print(f"Ensemble AUPRC (calibrated): {compute_auprc(val_labels, ens_cal):.4f}")
        print(f"Ensemble AUROC             : {compute_auroc(val_labels, ens_cal):.4f}")
    else:
        print(f"\nEnsemble AUPRC : {compute_auprc(val_labels, ens):.4f}")


In [ ]:
from sklearn.metrics import precision_recall_curve

calib_path = Path("checkpoints/track1/calibrator.pkl")

if fold_scores:
    if calib_path.exists():
        with open(calib_path, "rb") as f:
            _c = pickle.load(f)
        ens_plot = _c.transform(np.mean(fold_scores, axis=0))
    else:
        ens_plot = np.mean(fold_scores, axis=0)

    precision, recall, _ = precision_recall_curve(val_labels, ens_plot)
    ap = compute_auprc(val_labels, ens_plot)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(recall, precision, color="#c0392b", lw=2, label=f"Track 1  AUPRC = {ap:.4f}")
    ax.fill_between(recall, precision, alpha=0.15, color="#c0392b")
    ax.axhline(val_labels.mean(), ls="--", color="grey", lw=1, label="Baseline (prior)")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision\u2013Recall Curve \u2014 Track 1")
    ax.legend(fontsize=10)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig("logs/prcurve_track1.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("No Track 1 fold scores available — run training first.")


---
## 8 · Track 2 — EEG + Video

**Task:** Binary detection using both EEG and pre-extracted video features.  
**Key mechanism — Uncertainty-aware Gating:**

1. The EEG embedding is passed through a lightweight scorer **K times** with MC-Dropout active.
2. The variance across passes is converted to a predictive **entropy** score.
3. A soft gate `g ∈ [0, 1]` is derived: `g → 0` when EEG is confident; `g → 1` when uncertain.
4. Cross-attention (EEG queries video keys/values) is activated proportionally to `g`.

This mirrors clinical practice: the camera feed is consulted only when the EEG signal is ambiguous (e.g. movement artefacts).

In [ ]:
!python scripts/train_track.py \
    --track 2 \
    --config configs/track2.yaml \
    --backbone_ckpt {BACKBONE_CKPT}

In [ ]:
torch.manual_seed(0)
ckpt_path   = Path("checkpoints/track2/fold0.pt")
scaler_path = Path("checkpoints/track2/scalers_fold0.pkl")

assert ckpt_path.exists(), "Track 2 checkpoint not found — run training first."

scalers  = load_scalers(scaler_path)
vid_bbs  = cfg_t2.get("video", {}).get("active_backbones", [])
df_val_s = df_val.sample(min(400, len(df_val)), random_state=0).reset_index(drop=True)

loader = build_dataloader(
    df_val_s, Path("processed/features/eeg"), scalers,
    track=2, video_dir=Path("processed/features/video"),
    video_backbones=vid_bbs, batch_size=64, augment=False, shuffle=False,
)
m2 = build_model_from_config(cfg_t2).to(DEVICE)
m2.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False)["model_state"])

# MC-Dropout: BN uses stored running stats (eval); Dropout stays stochastic (train)
m2.eval()
for _mod in m2.modules():
    if isinstance(_mod, torch.nn.Dropout):
        _mod.train()

gates, labels = [], []
with torch.no_grad():
    for batch in loader:
        eeg = batch["eeg"].to(DEVICE)
        vf  = {k: v.to(DEVICE) for k, v in batch.get("video_features", {}).items()}
        out = m2(eeg, video_features=vf or None, mode="track2")
        gates.extend(out["gate"].cpu().tolist())
        labels.extend(batch["label"].tolist())

gates  = np.array(gates)
labels = np.array(labels)

print("Uncertainty gate statistics:")
print(f"  Positive samples  mean gate : {gates[labels == 1].mean():.3f}")
print(f"  Negative samples  mean gate : {gates[labels == 0].mean():.3f}")
print(f"  Overall           mean gate : {gates.mean():.3f}")
print(f"  (gate \u2192 1 = uncertain \u2192 video cross-attention activated)")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(gates[labels == 1], bins=30, alpha=0.6, color="#c0392b", label="Positive")
ax.hist(gates[labels == 0], bins=30, alpha=0.6, color="#2980b9", label="Negative")
ax.set_xlabel("Gate value")
ax.set_ylabel("Count")
ax.set_title("Uncertainty Gate Distribution \u2014 Track 2")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("logs/gate_distribution.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
import pickle

t2_fold_scores, t2_val_labels = [], None

for fold in range(5):
    ckpt_path_t2   = Path(f"checkpoints/track2/fold{fold}.pt")
    scaler_path_t2 = Path(f"checkpoints/track2/scalers_fold{fold}.pkl")
    if not ckpt_path_t2.exists():
        continue

    scalers_t2 = load_scalers(scaler_path_t2)
    vid_bbs_t2 = cfg_t2.get("video", {}).get("active_backbones", [])
    loader_t2  = build_dataloader(
        df_val, Path("processed/features/eeg"), scalers_t2,
        track=2, video_dir=Path("processed/features/video"),
        video_backbones=vid_bbs_t2, batch_size=256, augment=False, shuffle=False, num_workers=4,
    )
    m_t2 = build_model_from_config(cfg_t2).to(DEVICE)
    m_t2.load_state_dict(torch.load(ckpt_path_t2, map_location=DEVICE, weights_only=False)["model_state"])
    m_t2.eval()

    logits_t2, labels_t2 = [], []
    with torch.no_grad():
        for batch in loader_t2:
            eeg = batch["eeg"].to(DEVICE)
            vf  = {k: v.to(DEVICE) for k, v in batch.get("video_features", {}).items()}
            logits_t2.append(torch.sigmoid(m_t2(eeg, video_features=vf or None, mode="track2")["logit"]).cpu())
            labels_t2.append(batch["label"])

    s2 = torch.cat(logits_t2).numpy()
    t2 = torch.cat(labels_t2).numpy()
    if t2_val_labels is None:
        t2_val_labels = t2
    t2_fold_scores.append(s2)
    print(f"  Fold {fold}  AUPRC = {compute_auprc(t2, s2):.4f}")

if t2_fold_scores:
    ens_t2 = np.mean(t2_fold_scores, axis=0)
    calib_path_t2 = Path("checkpoints/track2/calibrator.pkl")
    if calib_path_t2.exists():
        with open(calib_path_t2, "rb") as f:
            calib_t2 = pickle.load(f)
        ens_t2_cal = calib_t2.transform(ens_t2)
        print(f"\nEnsemble AUPRC (raw)       : {compute_auprc(t2_val_labels, ens_t2):.4f}")
        print(f"Ensemble AUPRC (calibrated): {compute_auprc(t2_val_labels, ens_t2_cal):.4f}")
    else:
        print(f"\nEnsemble AUPRC : {compute_auprc(t2_val_labels, ens_t2):.4f}")
else:
    print("No Track 2 fold checkpoints found — run training first.")


---
## 9 · Track 3 — Localisation

**Task:** For **positive windows only**, classify the epileptogenic zone into one of 5 subtypes.  
**Metric:** Weighted-F1 (classes 1–5, weighted by support).  

**Architecture additions:**
- The dataloader automatically filters to `label == 1` for Track 3.
- Channel-level GAT features `(B, 29, 128)` are fed to a `ChannelTemporalAttn` module that captures the spatial propagation pattern of IEDs across electrodes.
- A `WeightedCELoss` with per-fold inverse-frequency weights and label smoothing addresses the class imbalance.
- `label_type` values 1–5 are remapped to 0–4 for `CrossEntropyLoss`.

In [ ]:
!python scripts/train_track.py \
    --track 3 \
    --config configs/track3.yaml \
    --backbone_ckpt {BACKBONE_CKPT}

In [ ]:
from src.evaluation.metrics import compute_weighted_f1
from src.data.preprocessing import load_scalers

df_val_pos      = df_val[df_val["label"] == 1].reset_index(drop=True)
t3_fold_logits  = []
t3_targets      = None

for fold in range(5):
    ckpt_path_t3   = Path(f"checkpoints/track3/fold{fold}.pt")
    scaler_path_t3 = Path(f"checkpoints/track3/scalers_fold{fold}.pkl")
    if not ckpt_path_t3.exists():
        continue

    scalers_t3 = load_scalers(scaler_path_t3)
    vid_bbs3   = cfg_t3.get("video", {}).get("active_backbones", [])
    loader3 = build_dataloader(
        df_val_pos, Path("processed/features/eeg"), scalers_t3,
        track=3, video_dir=Path("processed/features/video"),
        video_backbones=vid_bbs3, batch_size=64, augment=False, shuffle=False,
    )

    m3 = build_model_from_config(cfg_t3).to(DEVICE)
    m3.load_state_dict(torch.load(ckpt_path_t3, map_location=DEVICE, weights_only=False)["model_state"])
    m3.eval()

    fold_logits, fold_targets = [], []
    with torch.no_grad():
        for batch in loader3:
            eeg = batch["eeg"].to(DEVICE)
            vf  = {k: v.to(DEVICE) for k, v in batch.get("video_features", {}).items()}
            out = m3(eeg, video_features=vf or None, mode="track3")
            fold_logits.append(out["logit"].cpu())
            fold_targets.append(batch["label_type"])

    fold_logits_t = torch.cat(fold_logits)
    t3_fold_logits.append(fold_logits_t)
    if t3_targets is None:
        t3_targets = torch.cat(fold_targets).numpy()
    fold_preds = (torch.argmax(fold_logits_t, dim=-1) + 1).numpy()
    print(f"  Fold {fold}  Weighted-F1 = {compute_weighted_f1(t3_targets, fold_preds):.4f}")

if t3_fold_logits:
    ens_logits = torch.stack(t3_fold_logits).mean(0)
    preds_ens  = (torch.argmax(ens_logits, dim=-1) + 1).numpy()
    wf1 = compute_weighted_f1(t3_targets, preds_ens)
    print(f"\nEnsemble Weighted-F1 : {wf1:.4f}")
    print(f"Positive val samples : {len(t3_targets)}")

    comp = pd.DataFrame({
        "Predictions": pd.Series(preds_ens).value_counts().sort_index(),
        "Targets"    : pd.Series(t3_targets).value_counts().sort_index(),
    })
    comp.index.name = "Class"
    print(f"\n{comp.to_string()}")
else:
    print("No Track 3 fold checkpoints found — run training first.")


---
## 10 · Evaluation & Calibration

### Isotonic Regression (Track 1/2)

`OOFCalibrator` wraps `sklearn.isotonic.IsotonicRegression` and enforces the anti-leakage constraint: the calibrator is fitted exclusively on **out-of-fold scores** and never on the official validation set or the candidate set.

In [ ]:
from src.evaluation.metrics import OOFCalibrator, find_best_threshold
from sklearn.calibration import calibration_curve

# NOTE: Using simulated OOF scores for illustration.
# In production, replace with real per-fold (targets, scores) from train_track.py.
rng = np.random.default_rng(42)
calib_demo = OOFCalibrator()
all_t, all_s = [], []

for _ in range(5):
    t = rng.integers(0, 2, 500)
    s = np.where(t == 1, rng.beta(4, 2, 500), rng.beta(2, 4, 500))
    calib_demo.add_fold(t, s)
    all_t.extend(t.tolist())
    all_s.extend(s.tolist())

calib_demo.fit()
all_t  = np.array(all_t)
all_s  = np.array(all_s)
cal_s  = calib_demo.transform(all_s)

thresh, f1_opt = find_best_threshold(all_t, cal_s)

print(f"AUPRC before calibration : {compute_auprc(all_t, all_s):.4f}")
print(f"AUPRC after  calibration : {compute_auprc(all_t, cal_s):.4f}")
print(f"Optimal threshold (F1)   : {thresh:.3f}  \u2192  F1 = {f1_opt:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
fig.suptitle("Reliability Diagram — Isotonic Calibration", fontsize=12, fontweight="bold")

for ax, (scores, label, color) in zip(axes, [
    (all_s, "Before calibration",  "#2980b9"),
    (cal_s, "After calibration",   "#27ae60"),
]):
    frac, mean_p = calibration_curve(all_t, scores, n_bins=10)
    ax.plot(mean_p, frac,  "o-", color=color, lw=2, ms=5, label=label)
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Perfect")
    ax.set_xlabel("Mean predicted score")
    ax.set_ylabel("Fraction of positives")
    ax.set_title(label)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("logs/calibration_reliability.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 11 · Inference & Submission

`predict_candidate.py` enforces three invariants at inference time:

1. **`batch_size=1`** — no cross-sample statistics can leak through BatchNorm or the scaler.
2. **Scalers from the full training set** — not fold-specific, to avoid fold-selection bias.
3. **Isotonic calibration applied** if a fitted calibrator exists.

Fold predictions are averaged (Track 1/2) or majority-voted (Track 3) before writing the submission CSV.

In [ ]:
!python scripts/predict_candidate.py --track 1 --config configs/track1.yaml

In [ ]:
!python scripts/predict_candidate.py --track 2 --config configs/track2.yaml

In [ ]:
!python scripts/predict_candidate.py --track 3 --config configs/track3.yaml

In [ ]:
print(f"{'Track':<8} {'Samples':>8}  {'Score range / distribution':<40}  {'ZIP size':>10}")
print("-" * 72)

for track in [1, 2, 3]:
    csv_path = Path(f"submissions/track{track}_predictions.csv")
    zip_path = Path(f"submissions/track{track}_submission.zip")
    if not csv_path.exists():
        print(f"  T{track}     NOT GENERATED")
        continue
    sub = pd.read_csv(csv_path)
    col = "score" if track in (1, 2) else "prediction"
    if track in (1, 2):
        detail = f"[{sub[col].min():.3f}, {sub[col].max():.3f}]  mean={sub[col].mean():.3f}"
    else:
        detail = str(sub[col].value_counts().sort_index().to_dict())
    zip_sz = f"{zip_path.stat().st_size / 1024:.1f} KB" if zip_path.exists() else "missing"
    print(f"  T{track}     {len(sub):>8,}  {detail:<40}  {zip_sz:>10}")

---
## 12 · Unit Tests

The test suite (`tests/test_all.py`) covers every component:

| Class | Tests |
|:------|:------|
| `RMSNorm` | shape, no-NaN |
| `NativeMambaBlock` | shape, residual, no-NaN |
| `EEGMambaEncoder` | output shape, no-NaN, gradient flow |
| `DynamicGAT` | shape, no-NaN, layer residual |
| `EEGBackbone` | embed shape, channel feats, freeze/unfreeze, gradient flow |
| `NeuroMMModel` | all modes (track1/2/3/pretrain), backward, invalid mode |
| `Losses` | FocalLoss, PolyLoss, FocalPolyLoss, WeightedCE, MultiTask, backward |
| `Metrics` | AUPRC, Weighted-F1, OOFCalibrator, MetricTracker T1/T3 |
| `Augmentations` | SpecAugment, GaussianNoise, AugmentationPipeline |
| `Config` | load_config, build_model_from_config |

In [ ]:
!python -m pytest tests/ -v --tb=short -q 2>&1 | tail -40

---
## Pipeline Summary

```
prepare_data.py                →  annotations/  +  processed/features/
        ↓
pretrain.py                    →  checkpoints/pretrain/best_backbone.pt
        ↓
train_track.py  --track 1      →  checkpoints/track1/fold{0-4}.pt  +  calibrator.pkl
train_track.py  --track 2      →  checkpoints/track2/fold{0-4}.pt  +  calibrator.pkl
train_track.py  --track 3      →  checkpoints/track3/fold{0-4}.pt
        ↓
predict_candidate.py           →  submissions/track{1,2,3}_submission.zip
        ↓
Codabench upload               →  https://2026.neuromm.org
```

### Anti-leakage checklist

- `GroupKFold(subject_id)` — no patient appears in both train and validation folds  
- `RobustScaler` fitted on **fold-train only**  
- Isotonic calibration fitted on **OOF scores only**  
- Candidate inference at **`batch_size = 1`**  
- No global normalisation over the candidate set  

### References

- Mamba SSM — [Gu & Dao, 2023](https://arxiv.org/abs/2312.00752)  
- Focal Loss — [Lin et al., 2017](https://arxiv.org/abs/1708.02002)  
- PolyLoss — [Leng et al., 2022](https://arxiv.org/abs/2204.12511)  
- Dataset — [NeuroMM/NeuroMM-2026](https://huggingface.co/datasets/NeuroMM/NeuroMM-2026)  
- Leaderboard — [https://2026.neuromm.org](https://2026.neuromm.org)